# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Sheheryar-khan/FlyRank-ML-Internship-Starter-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content page along with its metadata and historical search and engagement performance.

I will use the starter dataset (content_refresh_anonymized.csv), which contains content metadata and 90-day performance metrics. If I move to the full warehouse later, I will primarily use dim_content and fact_content_daily_performance.

I will use the historical 90-day metrics provided in the starter dataset as the basis for my analysis and feature engineering.

I will rank content pages by their priority for review using a proxy target (needs_review) that represents whether a page appears to be a strong candidate for refresh based on historical performance signals.

I will not use client names, URLs, or any information from future time periods, since that could introduce data leakage or expose sensitive information.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())

df[[
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days"
]].head()

Rows: 30000
Unique content pages: 30000


,content_id,impressions_90d,clicks_90d,sessions_90d,content_age_days
0,content_304f48230142,3803,29,17,187
1,content_a1fb4e703a9e,15320,7,9,445
2,content_9aa793d4d895,12581,11,11,141
3,content_331d6c4de07b,11751,58,78,463
4,content_d99b7a2d90ca,19140,24,145,263


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
These fields will be used as input features for the model because they describe the historical performance and characteristics of each content page.

- `impressions_90d`
- `clicks_90d`
- `ctr`
- `avg_position`
- `engagement_rate`
- `sessions_90d`
- `content_age_days`
- `days_since_last_update`
- `word_count`
- `search_volume`
- `competition`

---

### Label (Proxy)
The starter dataset does not contain a direct target indicating whether a page needs to be refreshed. I will create a proxy label called:

- `needs_review` *(to be created)*

This proxy will be based on historical performance signals and business rules developed later in the project.

---

### Context
These fields provide additional information for understanding and explaining the recommendations but are not the primary prediction target.

- `content_id`
- `content_type`
- `main_intent`
- `age_tier`
- `freshness_tier`
- `trend_direction`

---

### Excluded
These fields will not be used because they do not directly support the content refresh objective or could introduce unnecessary bias.

- `client_id` – Identifier only; not useful as a predictive feature.
- `provider_used` – Describes how content was created rather than how it performs.
- `model_used` – Related to content generation, not refresh priority.
- `ai_sessions_90d` – Sparse data and not central to this lane.
- `ai_traffic_pct` – Sparse AI referral signal and outside the scope of this project.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
print("Total rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())


Total rows: 30000
Unique content pages: 30000


In [4]:
lane2_df = df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "content_age_days",
        "days_since_last_update",
        "ctr"
    ]
]

print("Rows in Lane 2 slice:", len(lane2_df))
print("Columns:", list(lane2_df.columns))

Rows in Lane 2 slice: 30000
Columns: ['content_id', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.